# DXY M/C/V Pattern Exploration

Reproducible analysis of M / C / V shape patterns on Welles Wilder RSI(14) for DXY.

**Sections:**
1. Data load + sanity checks
2. RSI(14) computation
3. Pattern detection
4. H1 — detector quality vs. manual labels
5. H2 — fractal self-similarity across timeframes
6. H3 — conditional forward returns
7. H4 — state-transition Markov + HMM comparison
8. Summary

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve() / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rsi_pattern import data, indicators, patterns, validate, forecast

## 1. Data load

Drop your BarChart CSVs at the paths shown below before running.

In [ ]:
for tf, path in data.expected_csv_paths().items():
    print(f'{tf:6s} -> {path}  (exists: {path.exists()})')

In [ ]:
dxy = {}
for tf in ['daily', '4h', '1h', '5m']:
    try:
        dxy[tf] = indicators.add_rsi(data.load_dxy(tf))
        print(f'{tf:6s} -> {len(dxy[tf])} bars, {dxy[tf].index[0]} to {dxy[tf].index[-1]}')
    except FileNotFoundError as e:
        print(f'{tf:6s} -> SKIPPED ({e})')

## 2. RSI plot + 3. Pattern detection

Visualize the RSI on the 1h timeframe with M / V regions highlighted.

In [ ]:
tf = '1h'
df = dxy[tf]
df = patterns.detect_all(df)

tail = df.tail(500)
fig, (ax_price, ax_rsi) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax_price.plot(tail.index, tail['close'], color='#888')
ax_price.set_ylabel('DXY close')

ax_rsi.plot(tail.index, tail['rsi14'], color='#444')
ax_rsi.axhline(70, color='red', linestyle='--', linewidth=0.5)
ax_rsi.axhline(50, color='gray', linestyle='--', linewidth=0.5)
ax_rsi.axhline(30, color='green', linestyle='--', linewidth=0.5)
ax_rsi.set_ylabel('RSI(14)')
for state, color in [('M', '#fb6'), ('V', '#6cf')]:
    mask = tail['state'] == state
    ax_rsi.fill_between(tail.index, 0, 100, where=mask, alpha=0.2, color=color, label=state)
ax_rsi.legend()
plt.suptitle(f'DXY {tf} — RSI(14) with M/V labels')
plt.tight_layout()
plt.show()

In [ ]:
patterns.summarize(df['state'])

## 5. H2 — Fractal self-similarity

In [ ]:
states_by_tf = {}
for tf, df in dxy.items():
    states_by_tf[tf] = patterns.detect_all(df)['state']
fractal_result = validate.fractal_compare(states_by_tf)
print('Occupancy (% bars) by timeframe:')
print(fractal_result['occupancy'])
print()
print('Duration KS tests (calendar-time):')
for k, v in fractal_result['duration_ks'].items():
    print(f'  {k}: stat={v["statistic"]:.3f} p={v["pvalue"]:.4f}')
print()
print('Transition matrix chi-squared:')
for k, v in fractal_result['transition_chi2'].items():
    print(f'  {k}: chi2={v["chi2"]:.2f} p={v["pvalue"]:.4f}')

## 6. H3 — Conditional forward returns

In [ ]:
fwd = validate.conditional_forward_returns(dxy['1h'], states_by_tf['1h'])
fwd

## 7. H4 — Markov + HMM

In [ ]:
mk = forecast.markov_baseline(states_by_tf['1h'])
print('Markov transition matrix:')
print(mk['transition_matrix'].round(3))
print(f'\nLog-likelihood: {mk["log_likelihood"]:.1f}, n_transitions={mk["n_transitions"]}')

hm = forecast.hmm_fit(states_by_tf['1h'], n_hidden=3)
print(f'\nHMM log-likelihood (3 hidden states): {hm["log_likelihood"]:.1f}')

## 8. Summary

Fill in once the analysis is run end-to-end.